# Phase 2: Model Training

This notebook trains the multi-temporal conditioned diffusion model. It is designed to run efficiently on a single Google Colab T4 GPU (16GB VRAM) using mixed precision (fp16) and gradient checkpointing.

**Core Methodology:**
1. Load pretrained `stable-diffusion-2-inpainting`.
2. Freeze the VAE and U-Net Encoder.
3. Add `TemporalConditioningLayer` blocks to the U-Net Decoder.
4. Fine-tune using a combined objective: L1 (masked) + SSIM (global) + SAM (masked).

In [ ]:
!pip install -r ../requirements.txt -q
import sys; sys.path.append('..')
import torch
from src.utils.config import default_config
from src.training.trainer import Trainer

## 1. Verify Environment & Config

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

cfg = default_config
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
cfg.batch_size = 4  # Adjust based on VRAM (4 fits on T4 with fp16)
cfg.num_epochs = 50
print(f"Training patches dir: {cfg.processed_patches}")

## 2. Initialize Trainer
The trainer handles dataset loading, model initialization, the mixed-precision training loop, and checkpointing.

In [ ]:
trainer = Trainer(config=cfg)

## 3. Run Training
If running in Colab, mount Google Drive and pass `gdrive_backup_dir` to automatically back up checkpoints every 5 epochs and save the best model. If the session crashes, re-running this cell will resume from the last checkpoint.

In [ ]:
# Mount Drive (Colab only)
# from google.colab import drive
# drive.mount('/content/drive')
# backup_path = "/content/drive/MyDrive/lissclear_checkpoints"

trainer.fit(
    gdrive_backup_dir=None # Replace with backup_path if using Drive
)

## 4. Plot Training Curves

In [ ]:
from src.evaluation.visualizer import ResultVisualizer
import json
from IPython.display import Image, display

history_file = cfg.inpainting_ckpt / "training_history.json"
if history_file.exists():
    with open(history_file, 'r') as f:
        history = json.load(f)
    
    vis = ResultVisualizer(output_dir=cfg.base_dir / "outputs")
    vis.training_curves(history, save_name="training_curves_notebook")
    display(Image(filename=cfg.base_dir / "outputs/training_curves_notebook.png"))
else:
    print("Training history not found.")